# Importando Bibliotecas

In [1]:
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint

In [2]:
pip install tensorflow transformers pandas scikit-learn

# Introdução

Neste projeto, o objetivo foi treinar um modelo baseado na arquitetura **BERT** utilizando o framework **Keras** para detectar bots no Twitter. Utilizamos um dataset de detecção de bots, com informações como o texto do tweet, contagem de retweets, menções e outras características, e treinamos um modelo para classificar se um usuário é ou não um bot.

O **BERT (Bidirectional Encoder Representations from Transformers)** é um modelo poderoso de processamento de linguagem natural que entende o contexto bidirecional dos textos, o que o torna ideal para tarefas de classificação de texto.



# Modelo

A abordagem para construção do modelo incluiu os seguintes passos:

1. **Tokenização dos Tweets**: Usamos o `BertTokenizer` para transformar o texto em representações numéricas que o modelo BERT pode processar.
2. **Construção do Modelo**: Utilizamos o modelo pré-treinado `TFBertModel` para extrair as representações contextuais dos tweets e, em seguida, adicionamos camadas densas e uma camada de saída para realizar a classificação binária (bot ou não).
3. **Treinamento**: O modelo foi treinado por 3 épocas utilizando a função de perda de entropia cruzada binária e o otimizador Adam. Utilizamos o `ModelCheckpoint` para salvar o melhor modelo baseado na perda de validação.


In [23]:
class BotDetectionModel(tf.keras.Model):
    def __init__(self, bert_model_name="bert-base-uncased", max_len=128):
        super(BotDetectionModel, self).__init__()
        self.max_len = max_len
        self.bert_model = TFBertModel.from_pretrained(bert_model_name)
        self.dropout = Dropout(0.3)
        self.dense1 = Dense(128, activation='relu')
        self.dense2 = Dense(1, activation='sigmoid')

    def call(self, inputs):
        input_ids, attention_mask = inputs
        bert_output = self.bert_model(input_ids=input_ids, attention_mask=attention_mask)[1]
        x = self.dropout(bert_output)
        x = self.dense1(x)
        x = self.dropout(x)
        output = self.dense2(x)
        return output

def preprocess(df):
    # Pre-processamento básico
    df = df[['Tweet', 'Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'Bot Label']]
    df['Tweet'] = df['Tweet'].apply(lambda x: str(x))
    return df

def tokenize(df, tokenizer, max_len):
    # Tokenização dos tweets utilizando o tokenizer BERT
    return tokenizer(
        df['Tweet'].tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

def main():
    # Carregar dataset
    data = pd.read_csv('/content/drive/MyDrive/Modulo 11/bot_detection_data.csv')

    # Pré-processamento e tokenização
    data = preprocess(data)
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    X_tokenized = tokenize(data, tokenizer, max_len=128)

    input_ids = X_tokenized['input_ids']
    attention_mask = X_tokenized['attention_mask']
    y = data['Bot Label'].values

    # Divisão em treino e teste
    X_train_ids, X_test_ids, X_train_mask, X_test_mask, y_train, y_test = train_test_split(
        input_ids.numpy(), attention_mask.numpy(), y, test_size=0.2, random_state=42
    )

    # Convertendo de volta para tensores tf
    X_train_ids = tf.convert_to_tensor(X_train_ids, dtype=tf.int32)
    X_test_ids = tf.convert_to_tensor(X_test_ids, dtype=tf.int32)
    X_train_mask = tf.convert_to_tensor(X_train_mask, dtype=tf.int32)
    X_test_mask = tf.convert_to_tensor(X_test_mask, dtype=tf.int32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)
    y_test = tf.convert_to_tensor(y_test, dtype=tf.int32)

    # Criar o modelo
    model = BotDetectionModel()

    # Compilação do modelo
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Persistência do modelo
    checkpoint = ModelCheckpoint('bot_detection_model.keras', monitor='val_loss', save_best_only=True, mode='min')

    # Treinamento
    model.fit(
        [X_train_ids, X_train_mask], y_train,
        validation_data=([X_test_ids, X_test_mask], y_test),
        epochs=3, batch_size=16, callbacks=[checkpoint]
    )

if __name__ == "__main__":
    main()


<ipython-input-23-981d0783d19a>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Tweet'] = df['Tweet'].apply(lambda x: str(x))
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.se

Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 394s 151ms/step - accuracy: 0.4956 - loss: 0.6974 - val_accuracy: 0.4980 - val_loss: 0.6964
Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 423s 146ms/step - accuracy: 0.4963 - loss: 0.6957 - val_accuracy: 0.4983 - val_loss: 0.6977
Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 391s 150ms/step - accuracy: 0.5000 - loss: 0.6947 - val_accuracy: 0.5044 - val_loss: 0.6953


### Resultados de Treinamento:

O treinamento foi realizado por 3 épocas, e as métricas de desempenho (acurácia e perda) foram registradas ao longo do tempo. Abaixo estão os resultados para cada época:

```plaintext
Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 394s 151ms/step - accuracy: 0.4956 - loss: 0.6974 - val_accuracy: 0.4980 - val_loss: 0.6964

Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 423s 146ms/step - accuracy: 0.4963 - loss: 0.6957 - val_accuracy: 0.4983 - val_loss: 0.6977

Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 391s 150ms/step - accuracy: 0.5000 - loss: 0.6947 - val_accuracy: 0.5044 - val_loss: 0.6953


# Conclusão

O modelo foi treinado utilizando o BERT pré-treinado e camadas densas para classificação de bots no Twitter. No entanto, os resultados de validação mostram que a acurácia no conjunto de validação não apresentou grande melhora durante as 3 épocas, ficando em torno de 50%.

Isso indica que o modelo não conseguiu aprender a distinguir bem entre bots e não bots com os dados fornecidos. Algumas possíveis melhorias incluem:

- Ajuste de hiperparâmetros: Tentar diferentes configurações de taxa de aprendizado, tamanho do lote, ou usar mais épocas de treinamento.
- Uso de mais features: Explorar o uso de features adicionais do dataset, como contagem de retweets, menções e informações de verificação de usuários.
- Aumento de dados: O desempenho do modelo pode melhorar com mais dados ou com técnicas de aumento de dados (data augmentation).
Embora o desempenho inicial do modelo não tenha sido ideal, ele fornece uma base sólida para futuras melhorias e experimentações.

